In [19]:
import pandas as pd
df = pd.read_csv('../lu2_datasets/Leescatalogus_Leesmateriaal.csv', sep=';')

**Kolomnamen consistentie**

In [20]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

**Kolommen verwijderen**

In [21]:
# Kolommen verwijderen die geen aanvullende informatie bevatten
df = df.drop(columns=["unnamed:_3", "genre_kopie"])

# Materiaal_id verwijderen, omdat Cataloguscode dezelfde unieke ID bevat
df = df.drop(columns=["materiaal_id"])

**Tekstwaarden standaardiseren**

In [22]:
# Kleine letters en onnodige spaties verwijderen
df["type_materiaal"] = df["type_materiaal"].str.strip().str.lower()
df["administratieve_prioriteit"] = df["administratieve_prioriteit"].str.strip().str.lower()
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.strip().str.lower()
df["interessecluster"] = df["interessecluster"].str.strip().str.lower()

df["doelgroep"] = (
    df["doelgroep"]
    .str.replace(" ", "", regex=False)
    .str.lower()
)

df["doelgroep_extra"] = (
    df["doelgroep_extra"]
    .str.replace(" ", "", regex=False)
    .str.lower()
)

# Taal standaardiseren
df["taal"] = df["taal"].replace({
    "NED": "Nederlands",
    "NL": "Nederlands",
    "nederlands": "Nederlands"
})

# Interne batchcode standaardiseren
df["interne_batchcode"] = df["interne_batchcode"].replace({
    "batch-2": "B2"
})

**Namen en auteurs corrigeren**

In [23]:
# Auteursnamen corrigeren
df["auteur"] = df["auteur"].replace({
    "Noami Alderman": "Naomi Alderman",
    "H. Mulisch": "Harry Mulisch",
    "Annejet vd Zijl": "Annejet van der Zijl"
})

# Meerdere auteurs op dezelfde manier noteren
df["auteur"] = df["auteur"].str.replace(
    "Hans en Monique Hagen",
    "Hans Hagen; Monique Hagen",
    regex=False
)

df["auteur"] = (
    df["auteur"]
    .str.replace(r"\s+en\s+", "; ", regex=True)
    .str.replace(r"\s*&\s*", "; ", regex=True)
    .str.replace(r"\s*/\s*", "; ", regex=True)
    .str.replace(r"\s*,\s*", "; ", regex=True)
)

**Thema's, onderwerpen en trefwoorden opschonen**

In [24]:
# Bronnen en URL's verwijderen uit trefwoorden
df["trefwoorden"] = (
    df["trefwoorden"]
    .str.replace(r"\s*[,/|;]\s*", ";", regex=True)
    .str.strip()
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True)
    .replace(["", "nan"], pd.NA)
)

# Bronnen verwijderen uit onderwerpen
df["onderwerp"] = (
    df["onderwerp"]
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True)
    .replace(["", "nan"], pd.NA)
)

# Thema's opschonen
df["themas_boek"] = (
    df["themas_boek"]
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True, case=False)
    .str.replace(r"\s*[,/|;]\s*", ";", regex=True)
    .str.strip(" ;")
    .replace(["", "nan"], pd.NA)
)

# Komma's vervangen door puntkomma's
kolommen = ["themas_boek", "onderwerp", "trefwoorden"]

for kolom in kolommen:
    df[kolom] = df[kolom].str.replace(r",\s*", ";", regex=True)

**Numerieke waarden en datatypes opschonen**

In [25]:
# Aantal pagina's: negatieve en nulwaarden verwijderen
df["aantal_paginas"] = (
    df["aantal_paginas"]
    .mask(df["aantal_paginas"] <= 0)
    .astype("Int64")
)

# Geschatte leestijd naar integer
df["geschatte_leestijd_min"] = df["geschatte_leestijd_min"].astype("Int64")

# Percentage omzetten naar float
df["aandeel_lange_woorden_pct"] = (
    df["aandeel_lange_woorden_pct"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Onwaarschijnlijke gemiddelde woordlengtes als ontbrekend markeren
df.loc[df["gem_woordlengte"] > 15, "gem_woordlengte"] = pd.NA

**Categorieën samenvoegen en standaardiseren**

In [26]:
# Doelgroep en doelgroep_extra combineren
def combineer_doelgroep(rij):
    waarden = set()

    for kolom in ["doelgroep", "doelgroep_extra"]:
        waarde = rij[kolom]

        if pd.notna(waarde):
            waarden.update(waarde.lower().split("/"))

    volgorde = ["mbo", "nt2"]
    resultaat = [waarde for waarde in volgorde if waarde in waarden]

    return "/".join(resultaat) if resultaat else pd.NA

df["doelgroep"] = df.apply(combineer_doelgroep, axis=1)
df = df.drop(columns=["doelgroep_extra"])


# Type materiaal en genre aanpassen
nieuwe_genres = df["type_materiaal"].str.split(" - ").str[1]

is_leeg = df["genre"].isna()
staat_al_iets = df["genre"].notna() & (df["genre"] != "onbekend")

df.loc[
    df["type_materiaal"].str.contains("boek -", na=False) & staat_al_iets,
    "genre"
] = df["genre"] + ";" + nieuwe_genres

df.loc[
    df["type_materiaal"].str.contains("boek -", na=False) & is_leeg,
    "genre"
] = nieuwe_genres

materiaal_mapping = {
    "online artikel": "artikel",
    "krant (papier)": "krant",
    "digitale krant": "krant",
    "boek - thriller": "boek",
    "boek - roman": "boek",
}

df["type_materiaal"] = df["type_materiaal"].replace(materiaal_mapping)

# Beschikbaar formaat standaardiseren
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].replace("digitaal", "online")

formaat_is_onbekend = (
    (df["beschikbaar_formaat"] == "onbekend")
    | df["beschikbaar_formaat"].isna()
)

heeft_bron_url = (
    df["bron_url"].notna()
    & (df["bron_url"] != "onbekend")
    & (df["bron_url"] != "")
)

df.loc[formaat_is_onbekend & heeft_bron_url, "beschikbaar_formaat"] = "online"

**Ontbrekende en onjuiste gegevens corrigeren**

In [27]:
# Gegevensstatus standaardiseren
df["gegevensstatus"] = df["gegevensstatus"].replace({
    "Aangevuld (deels gesimuleerd)": "beschikbaar",
    "gecontroleerd": "niet beschikbaar",
    "In behandeling": "uitgeleend"
}).fillna("niet beschikbaar")


# Publicatiejaren corrigeren
df.loc[df["titel"] == "De roadtrip", "publicatiejaar"] = "2025"
df.loc[df["titel"] == "Mijn naam is Edino", "publicatiejaar"] = "2025"

df["publicatiejaar"] = df["publicatiejaar"].replace({
    "2099": pd.NA,
    "onbekend": pd.NA,
    "nan": pd.NA
})


# Ontbrekende doelgroep en niveaus aanvullen
boeken = [
    "Libello voor mannen",
    "Libello voor vrouwen",
    "De Startkrant",
    "De Startweekkrant"
]

df.loc[df["titel"].isin(boeken), "doelgroep"] = "nt2"
df.loc[df["titel"].isin(boeken), "niveau"] = "2F"

**Ontbrekende trefwoorden aanvullen vanuit omschrijvingen**

In [28]:
# Omschrijvingen zonder punt gebruiken om metadata aan te vullen
# mask = ~df["Korte_omschrijving"].str.contains(r"\.", na=False)

# kolommen = ["Themas_boek", "Onderwerp", "Trefwoorden"]

# for kolom in kolommen:
#     df.loc[mask, kolom] = (
#         df.loc[mask, kolom].fillna("")
#         + ";"
#         + df.loc[mask, "Korte_omschrijving"]
#     ).str.strip(";")

In [29]:
df["geschatte_leestijd_min"].describe()

count         274.0
mean     377.343066
std      263.985609
min             5.0
25%           171.0
50%           366.0
75%           587.5
max           946.0
Name: geschatte_leestijd_min, dtype: Float64

In [30]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})

display(overzicht)

Aantal rijen en kolommen: (283, 29)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
titel,277,0,str,6
auteur,229,7,str,47
korte_omschrijving,282,0,str,1
type_materiaal,6,1,str,276
niveau,4,2,str,277
themas_boek,92,183,str,8
genre,7,20,str,256
onderwerp,46,183,str,54
trefwoorden,92,183,str,8
taal,1,0,str,282


In [32]:
df.to_csv("Leescatalogus_Opgeschoond.csv", index=False)